# Kifaru — poaching detection model

Fine-tunes a YOLO nano model for the aerial surveillance MVP and exports it to
ONNX, which is what the FastAPI service loads in production.

**Run this on Colab with a GPU runtime** (`Runtime → Change runtime type → T4 GPU`).
The deployment server has no GPU, so training here and shipping weights is the
only workable split.

### Why the class set is what it is

A rhino in frame is not poaching. A *person* or *vehicle* near a rhino is. So the
model detects three things and the alerting rule lives above it:

| id | class | why |
|----|---------|-----------------------------------------|
| 0 | rhino | the asset being protected |
| 1 | person | primary threat indicator |
| 2 | vehicle | car/truck/motorcycle, collapsed |

> **The trap this avoids:** fine-tuning on rhino-only images rebuilds the
> detection head for a single class, and the model permanently loses COCO's
> `person` / `car` / `truck`. It would then be structurally incapable of
> detecting poaching. Keep people and vehicles in the label set.

## 1. Setup

The version is pinned deliberately — an unpinned reinstall weeks from now can
shift training defaults and make results non-reproducible.

In [ ]:
!pip install -q ultralytics==8.4.118 onnxruntime==1.20.1

import ultralytics

ultralytics.checks()

## 2. Mount Drive

Colab wipes `/content` when the session ends (~90 min idle, 12h hard cap). A
long training run that writes only to `/content` loses `best.pt` when the tab
disconnects. Everything below writes to Drive instead.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/kifaru")
RUNS_DIR = PROJECT_DIR / "runs"
WEIGHTS_DIR = PROJECT_DIR / "weights"
DATA_DIR = Path("/content/datasets")  # scratch: large, re-downloadable

for directory in (RUNS_DIR, WEIGHTS_DIR, DATA_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("checkpoints ->", RUNS_DIR)

## 3. Config

`imgsz=640` matches what the production ONNX graph will be frozen at — change it
here and the server-side preprocessing must change with it.

In [ ]:
CONFIG = {
    "base_model": "yolo26n.pt",  # nano: the only sane size for a 6-core CPU server
    "epochs": 100,               # with patience below, this stops early on plateau
    "patience": 20,
    "imgsz": 640,
    "batch": 16,
    "seed": 0,
}

CLASSES = {0: "rhino", 1: "person", 2: "vehicle"}

CONFIG

## 4. Inspect the base dataset

Confirm African Wildlife actually contains a rhino class before building
anything on top of it. This downloads ~1.5k images on first run.

In [ ]:
from ultralytics.data.utils import check_det_dataset

wildlife = check_det_dataset("african-wildlife.yaml")

print("classes:", wildlife["names"])
print("root:   ", wildlife["path"])

RHINO_ID = next(
    (i for i, name in wildlife["names"].items() if "rhino" in name.lower()), None
)
assert RHINO_ID is not None, "No rhino class — pick a different base dataset."
print(f"\nrhino is class {RHINO_ID} in the source dataset")

## 5. Build the merged dataset

African Wildlife supplies rhinos but **no people or vehicles**, so it cannot
train the poaching signal on its own. This cell remaps its rhino labels into our
class set and leaves a clearly marked slot for the threat imagery.

### What you must supply

Drop YOLO-format data into `drive/MyDrive/kifaru/custom/` as
`images/{train,val}` + `labels/{train,val}`, using **our** class ids (0/1/2).
Two sources worth combining:

- **`VisDrone.yaml`** (built into Ultralytics) — real aerial imagery with
  pedestrian/car/van/truck. Right viewpoint for the threat classes.
- **Your own drone footage**, annotated in Roboflow or CVAT. This is the only
  source of *rhinos seen from above*, and it is what will actually decide
  whether the model works in the field.

> **Domain gap warning:** African Wildlife is ground-level photography. A model
> trained solely on it will underperform badly on top-down drone frames —
> different silhouette, scale and background. Treat it as a starting point, not
> a finished training set.

In [ ]:
import shutil

MERGED = DATA_DIR / "kifaru-merged"
CUSTOM = PROJECT_DIR / "custom"  # your own annotated data, already in 0/1/2

# Source class id -> our class id. Anything absent is dropped, so buffalo,
# elephant and zebra fall away; add them here if you want them detected too.
REMAP = {RHINO_ID: 0}


def convert_split(src_root: Path, split: str, remap: dict[int, int]) -> int:
    """Copy one split into MERGED, rewriting label ids and dropping the rest."""
    src_images = src_root / "images" / split
    src_labels = src_root / "labels" / split
    if not src_images.is_dir():
        print(f"  (no {split} split at {src_images})")
        return 0

    dst_images = MERGED / "images" / split
    dst_labels = MERGED / "labels" / split
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)

    kept = 0
    for image_path in src_images.iterdir():
        if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        label_path = src_labels / f"{image_path.stem}.txt"
        lines = []
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                parts = line.split()
                if not parts:
                    continue
                source_id = int(parts[0])
                if source_id in remap:
                    lines.append(" ".join([str(remap[source_id]), *parts[1:]]))

        # Images with no surviving boxes are still useful: they teach the model
        # what a non-rhino scene looks like and suppress false positives.
        shutil.copy2(image_path, dst_images / image_path.name)
        (dst_labels / f"{image_path.stem}.txt").write_text(
            "\n".join(lines) + ("\n" if lines else "")
        )
        kept += len(lines)
    return kept


shutil.rmtree(MERGED, ignore_errors=True)
wildlife_root = Path(wildlife["path"])

print("African Wildlife -> merged:")
for split in ("train", "val"):
    print(f"  {split}: {convert_split(wildlife_root, split, REMAP)} rhino boxes")

if CUSTOM.is_dir():
    print("\nYour custom data -> merged:")
    for split in ("train", "val"):
        # Already in our id space, so the remap is an identity mapping.
        n = convert_split(CUSTOM, split, {i: i for i in CLASSES})
        print(f"  {split}: {n} boxes")
else:
    print(f"\n!! No custom data at {CUSTOM}")
    print("!! Training will see ZERO person/vehicle examples, so those classes")
    print("!! cannot be learned. Fine for a first pipeline run; not shippable.")

In [ ]:
import yaml

data_yaml = MERGED / "kifaru.yaml"
data_yaml.write_text(
    yaml.safe_dump(
        {
            "path": str(MERGED),
            "train": "images/train",
            "val": "images/val",
            "names": CLASSES,
        },
        sort_keys=False,
    )
)
print(data_yaml.read_text())

### Class balance

Check this before training. If one class has an order of magnitude fewer boxes
than another, the model will largely ignore it and mAP will still look fine.

In [ ]:
from collections import Counter

for split in ("train", "val"):
    counts = Counter()
    label_dir = MERGED / "labels" / split
    if not label_dir.is_dir():
        continue
    for label_file in label_dir.glob("*.txt"):
        for line in label_file.read_text().splitlines():
            if line.strip():
                counts[CLASSES[int(line.split()[0])]] += 1
    total = sum(counts.values()) or 1
    print(f"{split}: {dict(counts)}")
    for name, n in counts.items():
        print(f"    {name:<8} {n:>6}  ({100 * n / total:.1f}%)")

## 6. Train

Resumable: if the Colab session drops, re-running picks up from the last Drive
checkpoint rather than starting over.

In [ ]:
from ultralytics import YOLO

RUN_NAME = "kifaru-v1"
last_checkpoint = RUNS_DIR / RUN_NAME / "weights" / "last.pt"

if last_checkpoint.exists():
    print(f"Resuming from {last_checkpoint}")
    model = YOLO(str(last_checkpoint))
    results = model.train(resume=True)
else:
    model = YOLO(CONFIG["base_model"])
    results = model.train(
        data=str(data_yaml),
        epochs=CONFIG["epochs"],
        patience=CONFIG["patience"],
        imgsz=CONFIG["imgsz"],
        batch=CONFIG["batch"],
        seed=CONFIG["seed"],
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        plots=True,
    )

## 7. Validate

Your current notebook has no metrics at all, so there is no way to tell whether
a retrain helped. Record these numbers for every run.

`mAP50-95` is the headline. **Per-class recall matters more here** — a missed
poacher is far worse than a false alarm a ranger dismisses.

In [ ]:
best_weights = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
model = YOLO(str(best_weights))
metrics = model.val(data=str(data_yaml), imgsz=CONFIG["imgsz"])

print(f"\nmAP50-95: {metrics.box.map:.4f}")
print(f"mAP50:    {metrics.box.map50:.4f}\n")

for i, name in CLASSES.items():
    try:
        p, r, ap50, ap = metrics.box.class_result(i)
        print(f"{name:<8} precision={p:.3f} recall={r:.3f} mAP50={ap50:.3f}")
    except (IndexError, KeyError):
        print(f"{name:<8} — no validation examples")

## 8. Export to ONNX

The production image serves ONNX rather than PyTorch. On a 6-core shared CPU
that is the difference between a ~250MB image starting in ~1s and a ~2GB image
starting in 10–20s, with faster inference besides.

`opset=12` is chosen for broad `onnxruntime` compatibility; `simplify` folds
constants so the server does less work per frame.

In [ ]:
onnx_path = Path(
    model.export(format="onnx", opset=12, simplify=True, imgsz=CONFIG["imgsz"])
)

destination = WEIGHTS_DIR / "best.onnx"
shutil.copy2(onnx_path, destination)
shutil.copy2(best_weights, WEIGHTS_DIR / "best.pt")  # keep for future re-exports

print(f"{destination}  ({destination.stat().st_size / 1e6:.1f} MB)")

## 9. Sanity-check the ONNX graph

Loads the exported file the same way the FastAPI service will — via
`onnxruntime`, no Ultralytics involved. If this cell fails, the model is not
deployable no matter how good the metrics were.

In [ ]:
import numpy as np
import onnxruntime as ort

session = ort.InferenceSession(str(destination), providers=["CPUExecutionProvider"])

input_meta = session.get_inputs()[0]
print(f"input : {input_meta.name} {input_meta.shape} {input_meta.type}")
for output in session.get_outputs():
    print(f"output: {output.name} {output.shape}")

dummy = np.zeros((1, 3, CONFIG["imgsz"], CONFIG["imgsz"]), dtype=np.float32)
prediction = session.run(None, {input_meta.name: dummy})[0]

print(f"\nforward pass OK -> {prediction.shape}")
print(f"expect 4 box coords + {len(CLASSES)} class scores = {4 + len(CLASSES)} rows")

## 10. Rough CPU timing

Colab's CPU is a reasonable stand-in for the server. Multiply by ~1.5–2× for
contention: the deployment box runs 22 containers on 6 shared cores.

Use the per-frame figure to pick a sampling stride. At 30fps, analysing every
frame of a 5-minute clip is 9,000 inferences — far past the 600s nginx timeout.
Sampling ~2 frames/second cuts that 15× and loses nothing operationally, since
poaching activity does not vanish within 500ms.

In [ ]:
import time

for _ in range(3):  # warm up
    session.run(None, {input_meta.name: dummy})

start = time.perf_counter()
runs = 20
for _ in range(runs):
    session.run(None, {input_meta.name: dummy})
per_frame = (time.perf_counter() - start) / runs

print(f"{per_frame * 1000:.1f} ms/frame on Colab CPU")
print(f"~{per_frame * 2000:.0f} ms/frame estimated on the server\n")
for minutes in (1, 5):
    frames = minutes * 60 * 2  # sampling at 2 fps
    print(f"{minutes}min clip @2fps = {frames} frames ~ {frames * per_frame * 2:.0f}s")

## Next steps

1. Download `best.onnx` from `Drive/MyDrive/kifaru/weights/` into the API repo at
   `models/best.onnx`.
2. Add `onnxruntime` to `requirements.txt` — **not** `ultralytics` or `torch`.
3. Replace `mock_predict` in `app/video_processor.py` with a real detector, and
   add frame sampling driven by the timing above.
4. Alerting rule, above the model: `person` or `vehicle` → alert; co-occurring
   with `rhino` → high severity. Explainable to a ranger in a way a bare
   confidence score is not.

**Before committing this notebook:** `Edit → Clear all outputs`. Training logs
and embedded plot images make diffs unreadable and bloat the repo.